# CausaSent — Kaggle demo notebook

Pulls the trained PhoBERT + mT5 checkpoints from HuggingFace and launches a public Gradio demo.

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (read-only is fine, models are public but rate limits are higher with a token).

**Pipeline:**
1. Clone the repo.
2. `pip install -r requirements.txt`.
3. Pull `Tamir39/causasent-phobert` and `Tamir39/causasent-mt5` into `checkpoints/`.
4. Launch Gradio with `--share` for a public `*.gradio.live` URL.


In [ ]:
# Cell 1 — Clone the repo
import os, subprocess

REPO_URL = 'https://github.com/tamir39/causa-sent.git'
REPO_DIR = '/kaggle/working/CausaSent'
BRANCH = 'feat/skeleton'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])

os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# Cell 2 — Install dependencies
%pip install -q -r requirements.txt 2>&1 | tail -10

In [ ]:
# Cell 3 — HF login (optional but reduces rate-limit warnings)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login OK')
except Exception as e:
    print(f'no HF_TOKEN secret found ({e}); proceeding unauthenticated')

In [ ]:
# Cell 4 — Pull trained checkpoints from HF Hub
from pathlib import Path
from huggingface_hub import snapshot_download

REPOS = {
    'Tamir39/causasent-phobert': 'checkpoints/phobert',
    'Tamir39/causasent-mt5':     'checkpoints/mt5',
}
for repo_id, local in REPOS.items():
    Path(local).mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=repo_id, local_dir=local, token=os.environ.get('HF_TOKEN'))
    print(f'pulled {repo_id} -> {local}')

# Sanity check
for p in ['checkpoints/phobert/best.pt', 'checkpoints/mt5/best.pt']:
    print(f'{p}: {Path(p).exists()}')

In [ ]:
# Cell 5 — Launch Gradio demo with public share link.
# Filtered output: only the public URL and tracebacks are shown.
# Full logs go to /tmp/causasent_demo.log.
!python -u -m src.demo.app --share 2>&1 | tee /tmp/causasent_demo.log | grep --line-buffered -E 'public URL|Running on local|Traceback|Error'